# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a guide for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("{}: {}".format(metadata.name, metadata.description))


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Show available RecordSets, Fields, and Columns by @id
record_sets = dataset.record_sets
print('Record Sets:')
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict): fields = [fields]
    print('  Fields:')
    for field in fields:
        print(f"    * Field @id: {field['@id']}, name: {field.get('name','N/A')}, dataType: {field.get('dataType','N/A')}")
        columns = field.get('column', [])
        if isinstance(columns, dict): columns = [columns]
        if columns:
            print('      Columns:')
            for col in columns:
                print(f"        - Column @id: {col['@id']}, name: {col.get('name','N/A')} (dataType: {col.get('dataType','N/A')})")

# For demonstration, print sample records for the first RecordSet
if record_sets:
    sample_rs_id = record_sets[0]['@id']
    print(f"\nSample records from RecordSet @id: {sample_rs_id}")
    for x in dataset.records(record_set=sample_rs_id):
        print(x)
        break   # print only one sample record for brevity

## 3. Data Extraction
Load data from specific record set(s) into DataFrames for analysis.
All entities are referenced by their `@id`.

In [ ]:
# Extract data from each RecordSet
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dfs = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dfs[record_set_id] = pd.DataFrame(records)

# Display available columns for the first DataFrame
primary_rs_id = record_set_ids[0] if record_set_ids else None
if primary_rs_id and primary_rs_id in dfs:
    print(f"Columns of RecordSet @id: {primary_rs_id}")
    print(dfs[primary_rs_id].columns.tolist())
    display(dfs[primary_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All field/column references are by their `@id`.

In [ ]:
# Choose the first numeric field (by @id) for demonstration
primary_df = dfs[primary_rs_id] if primary_rs_id and primary_rs_id in dfs else None

# Identify numeric field from metadata
numeric_field_id = None
group_field_id = None
for rs in dataset.record_sets:
    if rs['@id'] == primary_rs_id:
        for field in rs.get('field', []):
            # Pick first Integer or Float field
            dt = field.get('dataType','')
            if dt in ['schema:Integer', 'schema:Float', 'Integer', 'Float'] and not numeric_field_id:
                numeric_field_id = field['@id']
            # Pick a groupable field
            if dt in ['schema:Text','Text'] and not group_field_id:
                group_field_id = field['@id']

# If no numeric field is present, skip EDA
if primary_df is not None and numeric_field_id in primary_df.columns:
    threshold = primary_df[numeric_field_id].mean()
    # Filter records where numeric field > threshold
    filtered_df = primary_df[primary_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped (mean) of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found in the dataset or the DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
Use `@id` for field names.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_df is not None and numeric_field_id in primary_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(primary_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If a group_field is available, show boxplot
    if group_field_id and group_field_id in primary_df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=primary_df[group_field_id], y=primary_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Unable to visualize; numeric or group field missing.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Using `mlcroissant`, we loaded metadata and records directly from the FAIR^2 dataset schema.
- We reviewed the structure and fields (referenced by `@id`), and performed basic EDA, filtering, normalization, and grouping.
- Visualizations revealed the distribution of numeric variables and differences across groups (by `@id`).
- This workflow demonstrates FAIR, reproducible dataset access and processing, supporting further clinical or molecular analyses.
